# 03 — Terrain: slope, aspect, hillshade and the per-scene terrain-shadow flag (M1.3)

Copernicus GLO-30 (`cop-dem-glo-30`, a **DSM**: canopy and buildings included; heights in m above EGM2008) is resampled bilinearly to 30 m UTM
(EPSG:32644) on a grid that nests in the M1.2 10 m grid (ADR 0015). Slope, aspect (Horn) and the geometric terrain shadow are computed at 30 m and
replicated to 10 m. Shadow is computed **per scene** from that scene's own sun azimuth and elevation (stored by M1.2, ADR 0014) as self-shadow
(cos i <= 0) plus cast shadow (horizon-angle march), then aggregated per composite as the share of valid observations in shadow (ADR 0016).
All logic is in `highland.terrain`, `highland.terrain_checks` and `highland.viz3d`; this notebook orchestrates, tabulates and plots.

**3D is for viewing, never for measuring**: every number comes from the 2D outputs. Each 3D view states its vertical exaggeration (default 1x)
and is paired with the 2D map of the same layer. Hills hide what is behind them.

In [ ]:
import importlib.metadata as md_
import json
import os
import shutil
import subprocess
import sys
from pathlib import Path

# The kernel's working directory is notebooks/: find the repo root and make src/ importable
# without PYTHONPATH (the highland package is otherwise taken from the environment).
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
sys.path.insert(0, str(ROOT / "src"))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import rasterio

from highland import stac
from highland import terrain as T
from highland import terrain_checks as C
from highland.aoi import aoi_area_km2, load_aoi

pd.set_option("display.width", 220, "display.max_columns", 60, "display.max_rows", 100)
M12 = ROOT / "outputs" / "M1.2"
OUT = ROOT / "outputs" / "M1.3"
OUT.mkdir(parents=True, exist_ok=True)
FETCH_MTD = True  # download one MTD_TL.xml per scene for the D5 check (needs network; ~1-2 min)
NUM_WORKERS = 8

## 1. AOI and the M1.2 grid

The 10 m grid is rebuilt from the AOI and the frozen M1.1 items, and must equal the transform stored in the M1.2 meta files.

In [ ]:
aoi = load_aoi()
raw_items = stac.load_items(ROOT / C.M11_ITEMS)
items, _ = stac.dedupe_s2(raw_items)
geobox = stac.aoi_geobox(aoi, items, "s2", asset="B02", resolution=10.0)
meta_gb = C.geobox_from_meta(sorted((M12 / "meta").glob("s2_*.json"))[0])
assert geobox.transform == meta_gb.transform and tuple(geobox.shape) == tuple(meta_gb.shape)
print(
    f"AOI {aoi_area_km2(aoi):.1f} km2; M1.2 grid {tuple(geobox.shape)} px at {geobox.transform.a:g} m, corner "
    f"({geobox.transform.c:g}, {geobox.transform.f:g}), {geobox.crs}"
)

## 2. Run (resumable)

`run_all` loads the DEM (AOI-only first for the data-derived z_min, then with the horizon buffer), writes `dem_30m_buffer.tif` and
`terrain_30m.tif`, computes the shadow COGs for all 43 window-years from `outputs/M1.2/clear_masks/*.npz` and writes the summary tables.
`run_diagnostics` adds the MTD_TL sun-angle check (network), the SCL class 2 comparison, the illumination bins and the alignment shift.
Existing results are reused when their inputs are unchanged.

In [ ]:
res = C.run_all(ROOT, fetch_mtd=FETCH_MTD, workers=NUM_WORKERS)
diag = C.run_diagnostics(ROOT, fetch_mtd=FETCH_MTD, workers=NUM_WORKERS)
print("DEM items:", res["dem_items"])
for k in (
    "elev_min_deg",
    "elev_max_deg",
    "z_min_aoi_m",
    "z_max_aoi_m",
    "buffer_m_required",
    "buffer_cells",
    "buffer_m_used",
    "z_max_buffer_m",
    "aoi_only_vs_buffered_load_max_diff_m",
):
    print(f"{k:42s} {res[k]}")
print("slope (deg, AOI 30 m cells):", {k: round(v, 3) for k, v in res["slope_deg"].items()})
print("gdaldem hillshade vs highland.terrain.hillshade:", res["gdal_hillshade"])

## 3. Slope, aspect and hillshade (30 m, DSM-derived)

The 15.5 degree median slope in ADR 0009 is only a sanity reference (a different candidate-ranking computation). The DEM is a DSM: stand edges 20-30 m tall
create false slopes and stands opened since 2011-2015 still appear raised. Hillshade is display only (azimuth 315, altitude 45, GDAL's defaults).

In [ ]:
with rasterio.open(OUT / "terrain_30m.tif") as ds:
    terr = ds.read()
    names = ds.descriptions
    ext = (ds.bounds.left, ds.bounds.right, ds.bounds.bottom, ds.bounds.top)
fig, ax = plt.subplots(2, 2, figsize=(11, 10), constrained_layout=True)
for a, b, cm, lab in zip(
    ax.ravel(),
    terr,
    ["cividis", "cividis", "twilight", "gray"],
    [
        "elevation (m above EGM2008)",
        "slope (deg)",
        "aspect (deg, downslope bearing)",
        "hillshade az 315 alt 45 (display only)",
    ],
    strict=True,
):
    im = a.imshow(b, extent=ext, cmap=cm)
    a.set_title(lab, fontsize=10)
    fig.colorbar(im, ax=a, shrink=0.8)
fig.savefig(OUT / "maps_terrain.png", dpi=110)
plt.show()
sl = terr[1][np.isfinite(terr[1])]
plt.figure(figsize=(6, 3.5))
plt.hist(sl, bins=60)
plt.axvline(15.5, color="C3", ls="--", label="ADR 0009 median 15.5 (sanity reference)")
plt.axvline(np.median(sl), color="k", label=f"this run median {np.median(sl):.1f}")
plt.axvline(
    res["elev_min_deg"], color="C1", ls=":", label=f"min sun elevation {res['elev_min_deg']:.1f}"
)
plt.xlabel("slope (deg), 30 m Horn, AOI bbox")
plt.legend(fontsize=8)
plt.tight_layout()
plt.savefig(OUT / "slope_hist.png", dpi=110)
plt.show()

## 4. Sun geometry and the MTD_TL check (D5)

M1.2 stores the tile-mean sun angles from the STAC item (`s2:mean_solar_*`). The tile centre is about 40 km from the AOI, so the angles at the AOI can
differ slightly. `sun_grid_check.csv` compares them with the 5 km sun-angle grid in each scene's MTD_TL.xml, interpolated at the AOI centre and corners.
Criterion (plan D5): keep the tile mean if the largest angle difference, in slope-equivalent degrees, stays below the slope noise implied by the DEM
(sigma_slope ~ 0.433 sigma_z / res for Horn with uncorrelated errors, sigma_z = LE90 / 1.645).

In [ ]:
scenes = pd.read_csv(M12 / "scenes_used.csv")
print(
    scenes[["sun_azimuth_deg", "sun_elevation_deg"]].describe().loc[["count", "min", "50%", "max"]]
)
if (OUT / "sun_grid_check.csv").exists():
    sg = pd.read_csv(OUT / "sun_grid_check.csv")
    d_az = sg["aoi_centre_azimuth_deg"] - sg["csv_azimuth_deg"]
    d_zen = sg["aoi_centre_zenith_deg"] - sg["csv_zenith_deg"]
    print(
        "XML tile mean vs CSV (max abs, deg): az",
        float((sg["xml_mean_azimuth_deg"] - sg["csv_azimuth_deg"]).abs().max()),
        "zen",
        float((sg["xml_mean_zenith_deg"] - sg["csv_zenith_deg"]).abs().max()),
    )
    print(
        pd.DataFrame(
            {
                "AOI centre - tile mean, azimuth (deg)": d_az,
                "AOI centre - tile mean, zenith (deg)": d_zen,
            }
        )
        .describe()
        .loc[["min", "50%", "max"]]
    )
    print(
        "max |angle difference| at AOI centre/corners, slope-equivalent deg:",
        float(sg["slope_equiv_deg"].max()),
    )
    print(
        "variation across the AOI (max-min of 5 points): azimuth",
        float(sg["spread_az"].max()),
        "zenith",
        float(sg["spread_zen"].max()),
    )
    sig = [0.433 * (le90 / 1.645) / 30.0 for le90 in (2.0, 4.0)]
    print(
        "slope noise from DEM relative vertical accuracy (2 m, 4 m LE90), deg:",
        [round(float(np.degrees(np.arctan(s))), 2) for s in sig],
    )

## 5. Pilot: the lowest-sun scene

The scene with the lowest sun elevation is the hardest case for cast shadow. Self-shadow needs a slope above the sun elevation facing away; cast shadow needs a
horizon angle above it. Step convergence (D4) compares marching steps of 1, 0.5 and 0.25 cell.

In [ ]:
with rasterio.open(OUT / "dem_30m_buffer.tif") as ds:
    z = ds.read(1)
nb = res["buffer_cells"]
ZMIN_AOI = res[
    "z_min_aoi_buffered_load_m"
]  # minimum inside the AOI polygon (the guard concerns the AOI)
low = scenes.loc[scenes["sun_elevation_deg"].idxmin()]
print(low[["acq_id", "sun_azimuth_deg", "sun_elevation_deg"]].to_dict())
rows = []
for sf in (1.0, 0.5, 0.25):
    o = T.scene_shadow(
        z, 30.0, low.sun_azimuth_deg, low.sun_elevation_deg, nb, step_frac=sf, z_min=ZMIN_AOI
    )
    rows.append(
        {
            "step_cells": sf,
            "cells_self": int(o["self"].sum()),
            "cells_cast": int(o["cast"].sum()),
            "cells_any": int(o["any"].sum()),
            "share_any": float(o["any"].mean()),
            "cos_i_min": float(o["cos_i"].min()),
        }
    )
conv = pd.DataFrame(rows)
print(conv.to_string(index=False))
conv.to_csv(OUT / "step_convergence.csv", index=False)
o = T.scene_shadow(z, 30.0, low.sun_azimuth_deg, low.sun_elevation_deg, nb, z_min=ZMIN_AOI)
fig, ax = plt.subplots(1, 3, figsize=(13, 4.3), constrained_layout=True)
ax[0].imshow(o["cos_i"], cmap="cividis")
ax[0].set_title("cos i (lowest-sun scene)")
ax[1].imshow(o["self"] | o["cast"], cmap="gray_r")
ax[1].set_title(f"self or cast shadow: {int(o['any'].sum())} of {o['any'].size} cells")
ax[2].imshow(terr[3], cmap="gray")
ax[2].set_title("hillshade (display)")
fig.savefig(OUT / "pilot_lowest_sun.png", dpi=110)
plt.show()

## 6. All 43 window-years

Shares are of AOI pixels (10 m). `share_flag_ge_half` is the D6 flag (at least half of a pixel's valid observations shadowed);
`share_any_shadow` is any shadowed valid observation. `cos_i_*` are over each pixel's valid observations (cos i of the flat-ground reference at the sun elevation is sin(elevation)).
The thin-composite columns come from M1.2.

In [ ]:
summ = pd.read_csv(OUT / "shadow_summary.csv")
print(f"{len(summ)} window-years; scenes: {int(summ['n_scenes'].sum())}")
print(summ.to_string(index=False, float_format=lambda v: f"{v:.5g}"))
sc_tab = pd.read_csv(OUT / "scene_shadow.csv")
print(
    "per scene AOI share of self / cast / any shadow, max:",
    sc_tab[["share_self", "share_cast", "share_any"]].max().to_dict(),
)
print(
    "scenes with any modelled shadow in the AOI:",
    int((sc_tab["share_any"] > 0).sum()),
    "of",
    len(sc_tab),
)
fig, ax = plt.subplots(1, 2, figsize=(12, 3.8), constrained_layout=True)
lab = summ["season_year"].astype(str) + " " + summ["season"]
ax[0].bar(lab, summ["share_any_shadow"])
ax[0].set_title("AOI share with any shadowed valid observation")
ax[0].tick_params(axis="x", rotation=90, labelsize=6)
ax[1].plot(lab, summ["cos_i_mean_p01"], ".", label="p01")
ax[1].plot(lab, summ["cos_i_mean_p50"], ".", label="median")
ax[1].plot(lab, summ["cos_i_mean_p99"], ".", label="p99")
ax[1].set_title("cos i_mean over the AOI")
ax[1].legend()
ax[1].tick_params(axis="x", rotation=90, labelsize=6)
fig.savefig(OUT / "shadow_summary.png", dpi=110)
plt.show()

## 7. Diagnostics

SCL class 2 (the model's nearest counterpart) is compared with the modelled shadow at the 20 m SCL grid, split by processing-baseline era (before 04.00 class 2
also held dark soil and burns, ADR 0010/0012). This is a diagnostic, not a calibration. Then the illumination bins and the B08 versus cos i_mean shift check for SWM 2023 and NEM 2023, which is a **weak** alignment check: the cos i layer is 30 m blocks, a composite mixes many dates and sun angles, and a land-cover trend (forest on steep faces, tea/grass on gentle ground) dominates the correlation, so r changes monotonically with shift. The lake check below is the better one.

In [ ]:
cl2 = pd.read_csv(OUT / "scl2_vs_model.csv")
print(len(cl2), "scenes with SCL class 2 > 0")
print(
    cl2.groupby("era")[
        [
            "share_scl2",
            "share_model",
            "p_model_given_scl2",
            "p_scl2_given_model",
            "cos_i_mean_scl2",
            "cos_i_mean_clear",
        ]
    ]
    .agg(["count", "median", "max"])
    .T.to_string()
)
il = pd.read_csv(OUT / "illumination_bins.csv")
fig, ax = plt.subplots(1, 2, figsize=(11, 3.8), constrained_layout=True)
for w, g in il.groupby("window"):
    mid = (g["x_lo"] + g["x_hi"]) / 2
    ax[0].plot(mid, g["nir_b08_median"], "o-", label=w)
    ax[1].plot(mid, g["ndvi_median"], "o-", label=w)
ax[0].set_xlabel("cos_i_mean (decile bins)")
ax[0].set_ylabel("median B08 reflectance")
ax[1].set_xlabel("cos_i_mean (decile bins)")
ax[1].set_ylabel("median NDVI")
ax[0].legend()
fig.suptitle(
    "Illumination vs reflectance - confounded by land cover (not masked until M1.4)", fontsize=9
)
fig.savefig(OUT / "illumination_bins.png", dpi=110)
plt.show()
lk_json = OUT / "lake_alignment.json"
if not lk_json.exists():
    C.lake_alignment(ROOT, workers=NUM_WORKERS)  # ~1 min, network
lake = json.loads(lk_json.read_text())
print(
    "Lake check: DEM flattened-lake mask vs SCL class 6 frequency (baseline >= 04.00, clear_fraction > 0.5);",
    lake["n_scenes"],
    "scenes",
)
print(
    "Offset d: S2 water lies d east / north of the DEM lake. Step 10 m; mask 30 m, SCL 20 m, so resolution is about +-15 m."
)
for l_ in lake["lakes"]:
    print(json.dumps(l_))
fq = np.load(OUT / "water_frequency.npz")
lt = pd.read_csv(OUT / "lake_alignment.csv")
fig, ax = plt.subplots(1, 2, figsize=(10, 4.3), constrained_layout=True)
for a, li in zip(ax, (1, 2), strict=True):
    f10 = np.repeat(np.repeat(fq[f"freq20_lake{li}"], 2, 0), 2, 1)
    im = a.imshow(f10, cmap="Blues", vmin=0, vmax=1)
    a.contour(fq[f"mask10_lake{li}"], levels=[0.5], colors="C3", linewidths=1)
    a.contour(fq[f"flat10_lake{li}"], levels=[0.5], colors="k", linewidths=1, linestyles="--")
    a.set_title(
        f"lake {li}: S2 water frequency (10 m lattice) with the DEM lake (red) and flat core (dashed)",
        fontsize=8,
    )
    fig.colorbar(im, ax=a, shrink=0.8, label="water / clear observations")
fig.savefig(OUT / "lake_water_frequency.png", dpi=110)
plt.show()
al = pd.read_csv(OUT / "alignment_shift.csv")
for w, g in al.groupby("window"):
    best = g.loc[g["r"].abs().idxmax()]  # r is negative: strongest |r|, not signed max
    edge = abs(best.dy) == 3 or abs(best.dx) == 3
    print(
        "(B08 check: dy > 0 = row-positive = SOUTH, dx > 0 = east; the lake check uses dy > 0 = NORTH)"
    )
    print(
        f"{w}: r at (0,0) = {g[(g.dy == 0) & (g.dx == 0)].r.iloc[0]:.3f}; strongest |r| = {best.r:.3f} at shift (dy={int(best.dy)}, dx={int(best.dx)}) px"
        f"{' - ON THE EDGE of the search window: no interior maximum, check inconclusive' if edge else ''}"
    )
    print(g.pivot(index="dy", columns="dx", values="r").round(3))

## 8. Results with and without the flag

AOI median, p01 and p99 of NDVI, NBR and NDMI for all pixels, for D6-unflagged pixels and for pixels without any shadowed observation. The `cos_i_above_aoi_p10/p25`
rows drop the least-lit 10 / 25 % of AOI pixels per window and are a sensitivity check only, not a proposed cut. M1.6 and M1.7 carry both columns.

In [ ]:
ww = pd.read_csv(OUT / "with_without_flag.csv")
piv = ww.pivot_table(index=["season_year", "season", "index"], columns="subset", values="median")
print(
    "max |median(unflagged) - median(all)| per index:",
    (piv["unflagged"] - piv["all"]).abs().groupby("index").max().to_dict(),
)
print(
    "max |median(drop least-lit 10%) - median(all)| per index:",
    (piv["cos_i_above_aoi_p10"] - piv["all"]).abs().groupby("index").max().to_dict(),
)
print("max flagged share:", float(ww["flagged_share"].max()))
print(ww[(ww.season_year == 2023)].to_string(index=False, float_format=lambda v: f"{v:.4g}"))

## 9. 3D QA views (viewing only)

`highland.viz3d.drape` drapes a layer on the 30 m DEM grid (layers are resampled to the DEM grid first: bilinear for continuous values, nearest for classes).
Vertical exaggeration is **1x** unless stated on the view. PyVista lives in the optional `highland-3d` environment (ADR 0005); off-screen PNGs need a display or
`xvfb-run -a` (the conda-forge VTK has no OSMesa build). Without pyvista this section prints a note and is skipped. Each view is paired with its 2D map, which has colour bars or a legend. The flag view uses NEM 2018, a window that has flagged pixels (the flag is nearly empty elsewhere, ADR 0016).

In [ ]:
import rioxarray  # noqa: F401
import xarray as xr

from highland import viz3d

try:
    import pyvista  # noqa: F401

    HAVE_PV = True
except ImportError:
    HAVE_PV = False
    print(
        "pyvista not installed: skipping the 3D views (see ADR 0005 and docs/decisions/0015-*.md)."
    )
EXAG = 1.0
FLAG_NAMES = {0.0: "not flagged (share < 0.5)", 1.0: "flagged (share >= 0.5)"}
dem_da = rioxarray.open_rasterio(OUT / "terrain_30m.tif", masked=True)
dem = dem_da.isel(band=0, drop=True).rio.write_crs("EPSG:32644")
slope_da = dem_da.isel(band=1, drop=True).rio.write_crs("EPSG:32644")
cosm = rioxarray.open_rasterio(OUT / "s2_2018_NEM_shadow.tif", masked=True).isel(band=2, drop=True)
flag_nem = rioxarray.open_rasterio(OUT / "s2_2018_NEM_shadow.tif", masked=True).isel(
    band=0, drop=True
)
# NaN stays NaN (outside the AOI / no valid observation); 0 = not flagged, 1 = flagged
flag_nem = xr.where(
    flag_nem.isnull(), np.nan, (flag_nem >= T.SHADOW_SHARE_CUT).astype("float32")
).rio.write_crs("EPSG:32644")
rgb = (
    rioxarray.open_rasterio(M12 / "s2_2023_SWM_reflectance.tif", masked=True)
    .isel(band=[2, 1, 0])
    .rio.write_crs("EPSG:32644")
)
views = {
    "slope": (slope_da, "continuous", "slope (deg)", "cividis"),
    "truecolour_SWM2023": (rgb, "rgb", "true colour, SWM 2023 median", None),
    "cos_i_mean_NEM2018": (
        cosm.rio.write_crs("EPSG:32644"),
        "continuous",
        "cos i_mean, NEM 2018",
        "cividis",
    ),
    "flag_NEM2018": (flag_nem, "categorical", "D6 flag, NEM 2018", None),
}
if HAVE_PV:
    for name, (layer, kind, title, cmap) in views.items():
        kw = {"cmap": cmap} if cmap else {}
        if kind == "categorical":
            kw["class_names"] = FLAG_NAMES
        p = viz3d.drape(
            dem,
            layer,
            kind,
            exaggeration=EXAG,
            title=title,
            off_screen=True,
            screenshot=str(OUT / f"drape_{name}.png"),
            **kw,
        )
        p.close()
        print("saved", OUT / f"drape_{name}.png", f"(vertical exaggeration {EXAG:g}x)")

In [ ]:
from matplotlib.colors import ListedColormap
from matplotlib.lines import Line2D
from matplotlib.patches import Patch

# Each 2D panel shows the layer as the 3D view receives it: resampled onto the 30 m DEM grid (drape_arrays),
# with the same colour scale / true-colour stretch.
dem_ext = (
    float(dem.x.min()) - 15,
    float(dem.x.max()) + 15,
    float(dem.y.min()) - 15,
    float(dem.y.max()) + 15,
)
fig, ax = plt.subplots(1, 4, figsize=(19, 4.8), constrained_layout=True)
for a, (_name, (layer, kind, title, cmap)) in zip(ax, views.items(), strict=True):
    arrs = viz3d.drape_arrays(dem, layer, kind, EXAG)["scalars"]
    if kind == "rgb":
        u8, (lo, hi) = viz3d.rgb_stretch(arrs)
        a.imshow(u8, extent=dem_ext)
        sm = plt.cm.ScalarMappable(norm=plt.Normalize(lo, hi), cmap="gray")
        fig.colorbar(
            sm, ax=a, shrink=0.8, label=f"reflectance per band, p2-p98 stretch {lo:.3f}-{hi:.3f}"
        )
    elif kind == "categorical":
        fcm = ListedColormap([viz3d.OKABE_ITO[0], viz3d.OKABE_ITO[1]]).copy()
        fcm.set_bad("lightgrey")
        a.imshow(arrs, extent=dem_ext, cmap=fcm, vmin=0, vmax=1, interpolation="nearest")
        rr, cc = np.nonzero(arrs == 1)
        xs = dem_ext[0] + 30 * (cc + 0.5)
        ys = dem_ext[3] - 30 * (rr + 0.5)
        a.scatter(xs, ys, s=120, facecolors="none", edgecolors="k", lw=1.2)
        a.legend(
            handles=[
                Patch(color=viz3d.OKABE_ITO[0], label=FLAG_NAMES[0.0]),
                Patch(color=viz3d.OKABE_ITO[1], label=FLAG_NAMES[1.0]),
                Patch(color="lightgrey", label="no data (n_obs = 0 or outside the AOI)"),
                Line2D(
                    [],
                    [],
                    marker="o",
                    ls="",
                    mfc="none",
                    mec="k",
                    label=f"circle marks the {len(rr)} flagged 30 m cell(s) (enlarged)",
                ),
            ],
            loc="lower left",
            fontsize=7,
        )
    else:
        ccm = plt.get_cmap(cmap).copy()
        ccm.set_bad("lightgrey")
        im = a.imshow(arrs, extent=dem_ext, cmap=ccm)
        fig.colorbar(im, ax=a, shrink=0.8, label=title)
        if np.isnan(arrs).any() and "cos" in title:
            a.legend(
                handles=[Patch(color="lightgrey", label="no data (n_obs = 0 or outside the AOI)")],
                loc="lower left",
                fontsize=7,
            )
    a.set_title("2D: " + title, fontsize=9)
    a.set_xticks([])
    a.set_yticks([])
fig.savefig(OUT / "maps_for_3d_pairs.png", dpi=110)
plt.show()
if HAVE_PV:
    from IPython.display import Image, display

    display(Image(filename=str(OUT / "drape_slope.png")))

## 10. README with the resolved environment

In [ ]:
env_txt = None
mm = shutil.which("micromamba") or os.environ.get("MAMBA_EXE")
if mm:
    r = subprocess.run(
        [mm, "env", "export", "--explicit", "-p", sys.prefix], capture_output=True, text=True
    )
    if r.returncode == 0 and r.stdout.strip():
        env_txt = r.stdout
if env_txt is None:
    raise RuntimeError(
        "micromamba not found: run 'micromamba env export --explicit' and save it to outputs/M1.3/environment.explicit.txt"
    )
ENV_FILE = "environment-3d.explicit.txt" if HAVE_PV else "environment.explicit.txt"
(OUT / ENV_FILE).write_text(env_txt)
vers = {
    p: md_.version(p)
    for p in (
        "rasterio",
        "odc-stac",
        "odc-geo",
        "numpy",
        "xarray",
        "pystac-client",
        "planetary-computer",
    )
}
vers["GDAL"], vers["PROJ"] = rasterio.__gdal_version__, rasterio.__proj_version__
try:
    vers["pyvista"], vers["vtk"] = md_.version("pyvista"), md_.version("vtk")
except md_.PackageNotFoundError:
    vers["pyvista"] = "not in this env (3D PNGs come from the highland-3d env under xvfb-run)"
rt = json.loads((OUT / "run_timings.json").read_text())["timings"]
rt = {k: v for k, v in rt.items() if k != "per_window_seconds"}
dg = json.loads((OUT / "run_timings_diagnostics.json").read_text())
lk_secs = json.loads((OUT / "lake_alignment.json").read_text())["seconds"]
n_pngs = len(list(OUT.glob("drape_*.png")))
readme = f"""# outputs/M1.3 - terrain and per-scene terrain-shadow flag

Produced by `notebooks/03_terrain.ipynb` (code: `highland.terrain`, `highland.terrain_checks`, `highland.viz3d`), task M1.3. Git-ignored; rerun the
notebook to regenerate (resumable). ADRs: 0015 (DEM and derivatives), 0016 (shadow flag), 0014 amendment (what the M1.2 masks are), 0005 note (off-screen PNG).

**The DEM is a DSM** (Copernicus GLO-30, TanDEM-X 2011-2015: canopy and buildings included; heights in m above EGM2008, not ellipsoidal).
Slope here is canopy-surface slope, not ground slope. 3D views are for viewing, never for measuring; each states its vertical exaggeration ({EXAG:g}x).

## Inputs
- DEM items {res["dem_items"]}; bilinear to 30 m UTM nested in the M1.2 grid ({tuple(geobox.shape)} px at 10 m, corner {geobox.transform.c:g}, {geobox.transform.f:g}); buffer {res["buffer_cells"]} cells ({res["buffer_m_used"]:g} m).
- `outputs/M1.2/scenes_used.csv` and `clear_masks/*.npz` (per-scene valid masks and tile-mean sun angles, in scene order).

## Files
- `dem_30m_buffer.tif` (float32 m, buffered), `terrain_30m.tif` (bands dem, slope_deg, aspect_deg, hillshade_az315_alt45; NaN outer ring never reaches the AOI).
- `s2_<season_year>_<WIN>_shadow.tif` (10 m, float32): shadow_share (0-1), n_shadow, cos_i_mean, cos_i_min; NaN where n_obs = 0 or outside the AOI polygon. `_shadow.json` holds per-scene numbers and the resume key.
- `scene_shadow.csv`, `shadow_summary.csv`, `with_without_flag.csv`, `sun_grid_check.csv`, `scl2_vs_model.csv`, `illumination_bins.csv`, `alignment_shift.csv`, `step_convergence.csv`, `run_timings.json`, `run_timings_diagnostics.json`, `lake_alignment.{{csv,json}}`, `lake_alignment_bootstrap.csv`, `water_frequency.npz`, `lake_water_frequency.png`, PNGs ({n_pngs} 3D drapes `drape_*.png`).

## How the lake check works (`highland.terrain_checks.lake_alignment`)
The DEM flattens water bodies. Two lakes appear as zero-gradient cores in `terrain_30m.tif` (1244.0 m and 1231.5 m); the lake mask is the cells at that height joined to the core. It is correlated (Pearson r) with the
per-pixel frequency of SCL class 6 (water / clear observations, no cut) from the baseline >= 04.00 scenes with clear_fraction > 0.5, over +-60 m offsets in 10 m steps (`lake_alignment.csv`; dx > 0 east, dy > 0 NORTH). The SCL
is re-read in 800 m windows around each lake (the M1.1 cache cannot separate class 6) and stored in `water_frequency.npz`. `lake_alignment_bootstrap.csv` resamples scenes (300x). `lake_alignment.json` has the best offsets, the
threshold-free centroid and the centroids of frequency >= 0.3 / 0.5 / 0.7. The result is reported, not applied (ADR 0015/0016). The B08 versus cos_i_mean check in `alignment_shift.csv` is weak (dy > 0 = SOUTH there).

## How to run
`jupyter nbconvert --to notebook --execute notebooks/03_terrain.ipynb --output-dir <scratch>`; for the 3D PNGs use the `highland-3d` env under `xvfb-run -a` (needs a display or Xvfb). The notebook puts `<repo>/src` on `sys.path` itself, so no PYTHONPATH is needed
(the kernel's working directory is `notebooks/`). Network is needed for the DEM, MTD_TL and the lake SCL reads; `micromamba` must be on PATH for the environment export.

## Method (short)
Horn 3x3 slope/aspect at 30 m; cos i = unit normal . sun vector; self shadow = cos i <= 0; cast shadow = horizon-angle ray march (step 0.5 cell, DEM bilinear, no wrap)
with the buffer W = (2527 m - z_min,AOI) / tan(min sun elevation); aggregated per composite as shadowed valid observations / valid observations; flag = share >= 0.5 (ADR 0016).
Sun angles are the tile means from STAC per scene (D5 check in `sun_grid_check.csv`). Land cover is not masked (M1.4).

## Timings, memory and disk (4-core, 15 GB machine)
`run_timings.json` holds the full run of `run_all` (DEM and terrain rasters, all 43 window-years with force, per-window seconds, peak RSS); `run_timings_diagnostics.json` holds the MTD_TL and SCL class 2 timings; the lake check took {lk_secs} s (`lake_alignment.json`).
`run_all`: {json.dumps(rt)}
diagnostics: {json.dumps(dg)}
Disk: {sum(p.stat().st_size for p in OUT.rglob("*") if p.is_file()) / 1e6:.0f} MB in this directory.

Resolved environments (`micromamba env export --explicit`): {", ".join(p.name for p in sorted(OUT.glob("environment*.explicit.txt")))} (`environment.explicit.txt` = env `highland`, which computed everything except the 3D PNGs; `environment-3d.explicit.txt` = env `highland-3d` = environment.yml + environment-3d.yml, rendered under `xvfb-run -a`). Versions in the kernel that wrote this file: {json.dumps(vers)}.
"""
(OUT / "README.md").write_text(readme)
print(readme[:1500])